# 10 — Experiment Runner (Stage 2)

Thin launcher over `prjudge.judge`. All logic lives in the package; this notebook only configures, launches, and monitors runs. Resumability lives in the JSONL checkpoint — a dead kernel costs nothing, just re-run the cell.

**Workflow:** config sanity → pilot (tune the prompt) → dry-run (real APIs, cheap) → final battery. Pilot output is never mixed with `results_v2.jsonl`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))  # run from notebooks/
from prjudge.config import load_config
config = load_config()
from prjudge.judge import resolve_run_spec, run_judges
from prjudge.batch import run_judges_batch
print('config hash:', config.hash()[:16])
print('judges:', [j['name']+' -> '+j['model'] for j in config['judging']['judges']])
print('trials:', config['judging']['trials'], '| prompt:', config['judging']['prompt_version'])

config hash: 3e5de46e520aa5b5
judges: ['claude -> claude-sonnet-5', 'gpt -> gpt-5.6-terra']
trials: 3 | prompt: v4


## Config sanity
Confirm the selection manifest and variants are frozen before judging.

In [2]:
import json
sel_v = config['versions']['selection']
var_v = config['versions']['variants']
man = json.load(open(config.artifacts_dir / f'selection_manifest_{sel_v}.json'))
print('selected PRs:', man['n_selected'], '| composition:', man['composition'])
vdir = config.artifacts_dir / f'variants_{var_v}'
print('variant files:', len(list(vdir.glob('*__*.json'))) if vdir.exists() else 'MISSING — run 01_build_variants.py')

selected PRs: 30 | composition: {'difficulty': {'Type1_Direct': 30}, 'has_requested_changes': {'False': 26, 'True': 4}, 'language': {'Go': 5, 'JavaScript': 6, 'Python': 15, 'TypeScript': 4}, 'n': 30, 'n_repos': 18, 'n_short_desc': 2}
variant files: 240


## Pilot — prompt v4.1 stability check
Runs prompt **v4.1** (v4 with if/else-phrased items 2/7 and a "blocking" definition on item 9 — registered as its own version in `prompts.py`) on the same 12 v1∩v2 PRs (*baseline* × both judges × 3 trials ≈ 72 sync calls). The comparison cell below shows per-item flip rates for **v3 / v4 / v4.1** side by side — v3 from `results_v1.jsonl`, v4 from `pilot_prompt_v4.jsonl`, both already on disk.

In [3]:
import json
_v1 = set(json.load(open(config.artifacts_dir / 'selection_manifest_v1.json'))['task_ids'])
_v2 = set(json.load(open(config.artifacts_dir / 'selection_manifest_v2.json'))['task_ids'])
pilot_prs = sorted(_v1 & _v2)
print(len(pilot_prs), 'overlap PRs:', pilot_prs)

pilot = resolve_run_spec(config, run_name='pilot_prompt_v4.1',
                         variants=['baseline'], prs=pilot_prs,
                         trials=3, prompt_version='v4.1')
print('spec:', len(pilot.cells()), 'cells,', 'prompt', pilot.prompt_version)
summary = run_judges(config, pilot, mock=False)   # ~72 sync calls
print(summary)

12 overlap PRs: ['coreos-assembler__4359', 'dask__12221', 'mycli__1517', 'openbao__1906', 'pipecat__2735', 'protocompile__630', 'qutebrowser__8845', 'stylelint__8953', 'stylelint__9026', 'vega__4219', 'vitest__9521', 'zod__5672']
spec: 72 cells, prompt v4.1
  [25/72] done=0 ok=25 fail=0
  [50/72] done=0 ok=50 fail=0
RunSummary(run_name='pilot_prompt_v4.1', total_cells=72, already_done=0, attempted=72, succeeded=72, failed=0, input_tokens=224661, output_tokens=110586, errors=[])


### Per-item flip rate — v3 vs v4 vs v4.1
Flip rate = fraction of (PR, judge) cells whose 3 trial answers are **not** unanimous, per item. All three columns cover the same 12 PRs / baseline / both judges. v4.1 targets: items 2 and 7 (evidence/answer polarity slips) and 9 (blocking threshold) drop further; everything else holds.

In [4]:
import json, pandas as pd

def item_flip_rates(path, prompt_version, prs):
    rows = [json.loads(l) for l in open(path)]
    recs = []
    for r in rows:
        cl = (r.get('parsed') or {}).get('checklist') or {}
        if (r['variant'] != 'baseline' or r['task_id'] not in prs
                or r.get('prompt_version') != prompt_version or not cl):
            continue
        for i in range(1, 11):
            recs.append({'task': r['task_id'], 'judge': r['judge'], 'item': i,
                         'answer': cl[f'item_{i}']['answer']})
    df = pd.DataFrame(recs)
    per_cell = df.groupby(['task', 'judge', 'item'])['answer'].nunique()
    return per_cell.groupby('item').apply(lambda s: (s > 1).mean())

prs = set(pilot_prs)
runs_dir = config.artifacts_dir / 'runs'
cmp = pd.DataFrame({
    'v3': item_flip_rates(runs_dir / 'results_v1.jsonl', 'v3', prs),
    'v4': item_flip_rates(runs_dir / 'pilot_prompt_v4.jsonl', 'v4', prs),
    'v4.1': item_flip_rates(runs_dir / 'pilot_prompt_v4.1.jsonl', 'v4.1', prs),
}).round(3)
cmp.index = [f'item_{i}' for i in cmp.index]
print(cmp)
print('\nmean flip rate:', cmp.mean().round(3).to_dict())

            v3     v4   v4.1
item_1   0.500  0.375  0.375
item_2   0.458  0.250  0.417
item_3   0.083  0.042  0.042
item_4   0.042  0.000  0.000
item_5   0.000  0.083  0.042
item_6   0.000  0.000  0.000
item_7   0.208  0.250  0.167
item_8   0.083  0.083  0.000
item_9   0.500  0.375  0.458
item_10  0.458  0.375  0.458

mean flip rate: {'v3': 0.233, 'v4': 0.183, 'v4.1': 0.196}


## Dry-run — validate schema enforcement on real APIs (~$0.10)
2 PRs × 1 variant × 1 judge × 1 trial. Confirms structured output parses on the pinned provider before committing to the full battery.

In [ ]:
dry = resolve_run_spec(config, run_name='dry_run', variants=['baseline'],
                       judges=[config['judging']['judges'][0]['name']], limit=2, trials=1)
# run_judges(config, dry, mock=False)   # uncomment to spend ~$0.10
print('dry-run spec:', len(dry.cells()), 'cells')

## Final battery — full matrix, frozen prompt
30 × 8 × 2 × 3 = 1,440 calls → `artifacts/runs/results_v2.jsonl`. Resumable: re-run this cell after any interruption and it continues from the checkpoint.

In [ ]:
final = resolve_run_spec(config, run_name=config['judging']['final_run_name'])
print('final spec:', len(final.cells()), 'cells,', 'prompt', final.prompt_version)
# summary = run_judges(config, final, mock=False)   # uncomment for the real run
# print(summary)

## Final run (batch) — 50% cheaper, same models

Alternative to the cell above for the final battery (or any large extension-pool
run): submits/collects through each provider's Batch API instead of calling
synchronously. `run_judges_batch` is an idempotent *advance* step — re-run this
cell later to collect finished provider batches and submit whatever is still
missing; it reports `collected_now` / `in_flight` / `needs_sync` each time.
Writes to the exact same `{run_name}.jsonl` as the sync cell above, so the two
can be freely mixed (each row's `api_mode` records which path produced it).

In [ ]:
final = resolve_run_spec(config, run_name=config['judging']['final_run_name'])
print('final spec:', len(final.cells()), 'cells,', 'prompt', final.prompt_version)
# batch_summary = run_judges_batch(config, final, mock=False)   # uncomment; re-run cell later to collect
# print(batch_summary)

## Progress / cost summary

In [ ]:
from prjudge.judge import existing_keys
run = config['judging']['final_run_name']
path = config.artifacts_dir / 'runs' / f'{run}.jsonl'
done = existing_keys(path) if path.exists() else set()
print(f'{run}: {len(done)} / {len(final.cells())} cells complete')